## Predicting the NBA Finals MVP with ML
#### *by Noah Ford*

This notebook walks through the **analysis**: load already-scraped Finals box scores, build features, benchmark simple heuristics, then train a logistic model.

Scraping Basketball-Reference (HTML quirks, rate limits, four table types) lives in [`scripts/`](scripts/). To refresh raw data after a new Finals:

```bash
python3 scripts/refresh_data.py
```

HTML is an optional local cache (`data/series_html/finals/`, `data/series_html/meta/`, `data/teams/html/`) and is gitignored. The notebook reads **CSVs**.


#### Importing packages and helpers

Feature loading, MVP/champion lookups, and softmax helpers live in [`helpers/`](helpers/).

In [ ]:
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, classification_report
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler

from helpers import (
    ADVANCED_FEATURE_COLS,
    FINALS_SERIES_GAMES_CSV,
    FULL_TOP_8_CSV,
    FULL_TOP_8_UNRANKED_ADVANCED_CSV,
    FULL_TOP_8_UNRANKED_CSV,
    FINALS_MVP_CSV,
    CHAMPIONS_CSV,
    CHAMPIONS_SEASONS_CSV,
    LEAN_MODEL_FEATURE_COLS,
    SERIES_TABLES_WINNER_DIR,
    build_top8,
    champ_from_year,
    control_trial,
    latest_finals_year,
    list_winner_csvs,
    load_winner_csv,
    logo_from_team_name,
    mvp_from_year,
    normalize_player_name,
    select_model_features,
    softmax_shares,
    top_table_from_csv,
)

print("helpers ready; latest Finals year =", latest_finals_year())
print("advanced features:", ADVANCED_FEATURE_COLS)
print("lean model features:", LEAN_MODEL_FEATURE_COLS)
print("series lengths:", FINALS_SERIES_GAMES_CSV)


## The data layer

Everything below assumes these committed tables already exist:

| Path | Contents |
|------|----------|
| `data/series_tables/basic/winner/` | Champion Finals box scores (one CSV per series) |
| `data/series_tables/advanced/winner/` | Champion advanced tables (USG%, ORtg, DRtg, …; 1984+) |
| `data/meta/finals_mvp.csv` | Actual Finals MVP history |
| `data/meta/finals_series_games.csv` | Finals series length (games) per year |
| `data/meta/champions.csv` | Champion team names (newest-first, aligned with MVP file) |
| `data/teams/champions_seasons.csv` | Season summary + logo URLs for viz |

The logistic model uses **winner basic** stats plus **USG%, ORtg, DRtg** from winner advanced (years 1984–present, when BBR publishes advanced series tables).


In [ ]:
mvp_history = pd.read_csv(FINALS_MVP_CSV, index_col=0)
champions = pd.read_csv(CHAMPIONS_CSV, index_col=0)
print(f"{len(mvp_history)} Finals MVPs through {mvp_history['Season'].iloc[0]}")
mvp_history.head(3)


In [ ]:
winner_csvs = list_winner_csvs()
print(f"{len(winner_csvs)} champion box-score files")
sample = load_winner_csv(winner_csvs[0])
sample.head(5)


## Features

A big modeling choice: we can represent each player by **raw counting stats** or by **rank among teammates** (`PTS!`, `AST!`, …).

Relative ranks answer: given this roster, how distinctive was this player? Raw stats answer: how good was the line in absolute terms?

We keep both. The logistic model trains on the **unranked** top-8 table, with optional advanced rates (`USG%`, `ORtg`, `DRtg`) merged from the advanced series tables.


In [ ]:
# Rebuild from CSVs (fast, no network). Or load a prior build from output/.
FULL_TOP_8 = build_top8(rank=True)
FULL_TOP_8_unranked = build_top8(rank=False)
# Advanced box scores exist on BBR from 1984 on — require those years for a clean join.
FULL_TOP_8_unranked_advanced = build_top8(
    rank=False, advanced=True, require_advanced=True
)

FULL_TOP_8.to_csv(FULL_TOP_8_CSV)
FULL_TOP_8_unranked.to_csv(FULL_TOP_8_UNRANKED_CSV)
FULL_TOP_8_unranked_advanced.to_csv(FULL_TOP_8_UNRANKED_ADVANCED_CSV)

print(FULL_TOP_8.shape, "years", FULL_TOP_8["Year"].min(), "→", FULL_TOP_8["Year"].max())
print(
    "advanced:",
    FULL_TOP_8_unranked_advanced.shape,
    FULL_TOP_8_unranked_advanced["Year"].min(),
    "→",
    FULL_TOP_8_unranked_advanced["Year"].max(),
    "cols",
    list(ADVANCED_FEATURE_COLS),
)
FULL_TOP_8_unranked_advanced.sample(5, random_state=0)


In [ ]:
# Peek at one year of ranked features
FULL_TOP_8[FULL_TOP_8["Year"] == latest_finals_year()].head(8)


## Machine Learning

### Quick and dirty baselines

Before logistic regression: is the Finals MVP just the leader in points, or in points+rebounds+assists?

These heuristics use the same champion box scores — no model required.


In [ ]:
just_points_output = control_trial("pts")
ppg_trb_ast_output = control_trial("pra")

print("Just points:")
print(just_points_output["Correct"].value_counts(), "\n")
print("Points + rebounds + assists:")
print(ppg_trb_ast_output["Correct"].value_counts())
ppg_trb_ast_output.head(5)


In [ ]:
n = len(ppg_trb_ast_output)
print("PRA accuracy:", round(ppg_trb_ast_output["Correct"].mean(), 2), f"({ppg_trb_ast_output['Correct'].sum()}/{n})")
print("PTS accuracy:", round(just_points_output["Correct"].mean(), 2), f"({just_points_output['Correct'].sum()}/{n})")


### Logistic regression

We train on top-8 scorers per champion Finals roster with a **lean** feature set (drops collinear counting stats like `FG`/`FT`/`ORB`/`DRB`/`PF`/`G`), plus `USG%`, `ORtg`, and `DRtg` from the advanced table (1984+ only).

`GM` = games missed = `Series_G − G`, where series length lives in [`data/meta/finals_series_games.csv`](data/meta/finals_series_games.csv) (max `G` on the champion box score).

Class imbalance is real (~1 MVP per 8 rows), so each training fold uses **SMOTE**. Predictions are **out-of-fold** via 5-fold CV so every player gets a held-out score.

Below we fit the same lean pipeline **with** and **without** the advanced columns on the same 1984+ years, then keep the advanced model for `output/`.


In [ ]:
def fit_oof_logistic(players: pd.DataFrame):
    """5-fold OOF logistic: returns (output_df, oof_true_labels)."""
    X = players.drop(["Year", "Player", "mvp"], axis=1)
    y = players["mvp"].astype(int)

    clf = LogisticRegression(max_iter=1000)
    smote = SMOTE(sampling_strategy="minority", random_state=42)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scaler = StandardScaler()

    oof_prob = np.zeros(len(X))
    oof_logits = np.zeros(len(X))
    oof_true = np.zeros(len(X))

    for train_index, test_index in kf.split(X):
        X_tr, X_te = X.iloc[train_index], X.iloc[test_index]
        y_tr, y_te = y.iloc[train_index], y.iloc[test_index]

        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)
        X_res, y_res = smote.fit_resample(X_tr_s, y_tr)

        clf.fit(X_res, y_res)
        oof_prob[test_index] = clf.predict_proba(X_te_s)[:, 1]
        oof_logits[test_index] = clf.decision_function(X_te_s)
        oof_true[test_index] = y_te

    out = players[["Year", "Player", "mvp"]].reset_index(drop=True).copy()
    out["prob_mvp"] = np.round(oof_prob, 2)
    out["Binary"] = (oof_prob >= 0.5).astype(int)
    out["Correct"] = out["Binary"] == out["mvp"].astype(int)
    out.insert(1, "MVP", out["Year"].apply(mvp_from_year))
    out.insert(1, "Team", out["Year"].apply(champ_from_year))
    out["mvp_share"] = (
        pd.Series(oof_logits)
        .groupby(out["Year"].values)
        .transform(lambda s: softmax_shares(s.to_numpy()))
        .round(3)
        .to_numpy()
    )
    out.insert(
        out.columns.get_loc("mvp_share"),
        "Rank",
        out.groupby(["Year", "Team"])["mvp_share"]
        .rank(ascending=False, method="dense")
        .astype(int),
    )
    front = ["Year", "Team", "MVP", "Player", "mvp_share"]
    rest = [c for c in out.columns if c not in front]
    out = out[front + rest]
    return out, oof_true, clf, X, y


def summarize_accuracy(out: pd.DataFrame, label: str) -> None:
    picks = out.loc[out.groupby("Year")["mvp_share"].idxmax()]
    year_hits = int(
        (
            picks["Player"].map(normalize_player_name)
            == picks["MVP"].map(normalize_player_name)
        ).sum()
    )
    year_n = len(picks)
    y_true = out["mvp"].astype(int)
    y_pred = out["Binary"]
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    print(f"{label}")
    print(f"  player binary accuracy: {out['Correct'].mean():.3f}")
    print(f"  MVP recall / precision: {recall:.3f} / {precision:.3f}")
    print(f"  year argmax(mvp_share): {year_hits}/{year_n} ({year_hits / year_n:.3f})")


# Lean feature matrix: same 1984+ rows, with vs without advanced cols
players_basic = select_model_features(FULL_TOP_8_unranked_advanced, advanced=False)
players_adv = select_model_features(FULL_TOP_8_unranked_advanced, advanced=True)

out_basic, _, _, _, _ = fit_oof_logistic(players_basic)
out_adv, oof_binary, logistic_classifier, X, y = fit_oof_logistic(players_adv)

print("Lean features:", list(X.columns))
print("Same years:", players_adv["Year"].min(), "→", players_adv["Year"].max())
summarize_accuracy(out_basic, "WITHOUT USG%/ORtg/DRtg")
summarize_accuracy(out_adv, "WITH    USG%/ORtg/DRtg")
X.head(3)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=2022
)
smote = SMOTE(sampling_strategy="minority")
X_train_SMOTE, y_train_SMOTE = smote.fit_resample(X_train, y_train)
print("train labels:\n", y_train.value_counts().to_string())
print("after SMOTE:\n", y_train_SMOTE.value_counts().to_string())


Two outputs per player (from the **with-advanced** model):

1. **`prob_mvp` / `Binary`** — independent P(MVP) ∈ [0,1], thresholded at 0.5  
2. **`mvp_share` / `Rank`** — softmax of logits within each Finals year (shares sum to 1)


In [ ]:
# Persist the advanced-features model output
output = out_adv
players = players_adv
output.to_csv("output/machine_learning_output.csv", index=False)
output.head(8)


In [ ]:
output[(output["prob_mvp"] > 0.1) & (output["Team"] == "Chicago Bulls")]


In [ ]:
print(classification_report(oof_binary, output["Binary"]))


In [ ]:
# ROC on the last fold's held-out slice (illustrative)
RocCurveDisplay.from_estimator(
    logistic_classifier,
    scaler.transform(X.iloc[test_index]),
    y.iloc[test_index],
)


Overall accuracy looks high mostly because non-MVPs dominate. For the MVP class, **recall** (catching true Finals MVPs) is the number to watch — typically better than the PRA baseline, but not perfect.

Also: 100% may not be the goal. Voters can be wrong, and some awards (e.g. role-player MVPs) are inherently awkward for counting-stat models.


In [ ]:
print("Baseline (PRA) mistakes vs model scores")
mistakes = ppg_trb_ast_output[ppg_trb_ast_output["Correct"] == False]
mistakes = pd.merge(mistakes, output, on=["Year", "Player"], how="left")
mistakes = mistakes.drop(columns=["Team_y", "Correct_y", "Binary", "mvp", "MVP"], errors="ignore")
mistakes.head(10)


In [ ]:
ml_mistakes = output[output["Correct"] == False].reset_index(drop=True)
ml_mistakes.to_csv("output/machine_learning_mistakes.csv", index=False)
print(len(ml_mistakes), "row-level mistakes (includes false positives on non-MVPs)")
ml_mistakes.sample(min(5, len(ml_mistakes)), random_state=0)


### Visualizations

Logo URLs come from `champions_seasons.csv` (no HTML scrape required).

In [ ]:
picks = output[(output["Binary"] == 1) | (output["mvp"] == 1)].reset_index(drop=True)
visual = picks.drop(columns=["Binary"])
visual["logo_url"] = visual.apply(
    lambda r: logo_from_team_name(r["Team"], r["Year"]), axis=1
)
visual.to_csv("output/tableau_winners_and_picks.csv", index=False)
visual.head(10)


## Conclusion

Fun project — and a cleaner workflow when scraping stays out of the storytelling notebook.

### Next steps

1. Push the logistic model further (other imbalance remedies, feature sets, models).
2. Experiment with loser tables / more advanced rate stats.
3. Visualize MVP-share findings in Tableau (`output/machine_learning_output.csv`).

### Refreshing data after a new Finals

```bash
python3 scripts/refresh_data.py
```

Then re-run this notebook from the features section.
